# FABRIC Ceph Manager — Interactive GUI

This section adds a small widget-based interface for:
- **Users**: list, apply templated caps (single/multi subvol contexts), delete, export
- **CephFS subvolumes**: create/resize, info, exists, delete

> Fill the connection settings, click **Connect**, then use the tabs.

In [ ]:
import json

from click import group
from fabric_ceph_client.fabric_ceph_client import CephManagerClient, ApiError
from fabric_reports_client.reports_api import ReportsApi
from soupsieve import select

In [ ]:

CEPH_BASE_URL = "https://23.134.232.211"   
REPO_BASE_URL = "https://reports.fabric-testbed.net/reports"
TOKEN = "/Users/kthare10/work/id_token_prod.json"                         
VERIFY_TLS = False
VOL_NAME = "CEPH-FS-01"
DEFAULT_VOL_SIZE = 10 * 1024**3 # 10G
DEFAULT_MODE = "0777"

In [ ]:
reports_api =  ReportsApi(base_url=REPO_BASE_URL, token_file=TOKEN)
ceph_api = CephManagerClient(base_url=CEPH_BASE_URL, token_file=TOKEN, verify=VERIFY_TLS)

## List Cluster Info

In [ ]:
try:
    cluster = ceph_api.list_cluster_info()
    clusters = cluster.get('data')
    if clusters and len(clusters) > 0:
        print(json.dumps(clusters, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during list_cluster_info:", e)

In [ ]:
if clusters and len(clusters) > 0:
    for cluster in clusters:
        users = ceph_api.list_users(cluster=cluster.get('cluster'))
        users = users.get('data')
        print(f"Users in cluster: {cluster.get('cluster')}: ", len(users))
        print(json.dumps(users, indent=4, sort_keys=True))

In [ ]:
import json, re

def _unescape_keyring_blob(s: str) -> str:
    # export sometimes returns a JSON-escaped string; unescape if needed
    s = str(s)
    if (s.startswith('"') and s.endswith('"')) or ('\\n' in s or '\\"' in s):
        try:
            return json.loads(s)
        except Exception:
            return s
    return s

_caps_re = re.compile(r'^\s*caps\s+(\w+)\s*=\s*"([^"]+)"\s*$', re.M)

def extract_caps_by_entity(keyring_text: str) -> dict:
    """
    Parse lines like: caps mds = "allow rw fsname=FS path=/vol/..."
    Returns: {"mds": "...", "mon": "...", "osd": "..."} (subset)
    """
    out = {}
    for m in _caps_re.finditer(keyring_text):
        entity, cap = m.group(1).lower(), m.group(2).strip()
        out[entity] = cap
    return out


## Create volumes for active users

In [ ]:
response = reports_api.query_users(user_active=True, fetch_all=True)
active_users = response.get('data')
print("Total number of active users: ", len(active_users))

In [ ]:
for user in active_users:
    if "kthare" in user.get('bastion_login'):
        selected_user = user
        break


In [ ]:
selected_cluster = "asia"

In [ ]:
user_entity = f"client.{selected_user.get('bastion_login')}"

In [ ]:
sub_vol_name = selected_user.get('bastion_login')

In [ ]:
group_name = None

In [ ]:
try:
    # No group passed
    response = ceph_api.create_or_resize_subvolume(cluster=selected_cluster,
                                                   vol_name=VOL_NAME,
                                                   subvol_name=sub_vol_name,
                                                   size=DEFAULT_VOL_SIZE,
                                                   mode=DEFAULT_MODE)
    print(f"Subvolume {sub_vol_name} created successfully for {user_entity} on {selected_cluster}")
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during create_or_resize_subvolume:", e)

In [ ]:
try:
    response = ceph_api.export_users(cluster=selected_cluster, entities=[user_entity])
    key_rings = response.get('clusters', {}).get(selected_cluster, {})
    apply_changes = False
    if user_entity not in key_rings:
        tmpl_caps = [
            {"entity": "mon", "cap": "allow r fsname={fs}"},
            {"entity": "mds", "cap": "allow rw fsname={fs} path={path}"},
            {"entity": "osd", "cap": "allow rw tag cephfs data={fs}"},
            {"entity": "osd", "cap": "allow rw tag cephfs metadata={fs}"},
        ]
        apply_changes = True
    else:
        print(f"User entity: {user_entity} already exists")
        # TODO add to existing permissions
        blob = key_rings.get(user_entity)
        text = _unescape_keyring_blob(blob)
        caps_now = extract_caps_by_entity(text)
        mds_now = caps_now.get("mds", "")
        if sub_vol_name in mds_now:
            print(f"Subvolume {sub_vol_name} already exists in capabilities -  No changes needed")
        else:
            tmpl_caps = [
            {"entity": "mon", "cap": "allow r fsname={fs}"},
            {"entity": "osd", "cap": "allow rw tag cephfs data={fs}"},
            {"entity": "osd", "cap": "allow rw tag cephfs metadata={fs}"},
        ]
            if mds_now:
                # keep current exact MDS cap as a literal (no placeholders)
                tmpl_caps.append({"entity": "mds", "cap": mds_now})
            # add the new MDS clause via placeholders for the new context
            tmpl_caps.append({"entity": "mds", "cap": "allow rw fsname={fs} path={path}"})
            apply_changes = True

    if apply_changes:
        response = ceph_api.apply_user_for_multiple_subvols(
            cluster=selected_cluster,
            user_entity=user_entity,
            template_capabilities=tmpl_caps,
            contexts=[
                ("CEPH-FS-01", sub_vol_name, group_name),
            ]
        )
        print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during export_users:", e)

In [ ]:
try:
    keyring = ceph_api.export_users(cluster=selected_cluster, entities=[user_entity])
    print(json.dumps(keyring, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during export_users:", e)

In [ ]:
try:
    response = ceph_api.get_subvolume_info(cluster=selected_cluster, vol_name=VOL_NAME, subvol_name=sub_vol_name)
    print("Subvolume info retrieved")
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during get_subvolume_info:", e)

## Create volumes for active projects


In [ ]:
response =  reports_api.query_projects(project_active=True, fetch_all=True)
active_projects = response.get('data')
print("Total number of active projects: ", len(active_projects))

In [ ]:
for project in active_projects:
    if "FABRIC Staff" == project['project_name']:
        selected_project = project
        break

print("Selected project:", selected_project)

In [ ]:
group_name = selected_project.get('project_id')
print("Group name:", group_name)

In [ ]:
sub_vol_name = "ceph_vol_1234"
print("Subvol name:", sub_vol_name)

In [ ]:
try:
    response = ceph_api.create_or_resize_subvolume(cluster=selected_cluster,
                                                   vol_name=VOL_NAME,
                                                   subvol_name=sub_vol_name,
                                                   size=DEFAULT_VOL_SIZE,
                                                   group_name=group_name)
    print(f"Subvolume {sub_vol_name} created successfully for {selected_project}")
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during create_or_resize_subvolume:", e)

In [ ]:
response = reports_api.query_users(user_active=True, fetch_all=True, project_id=selected_project.get('project_id'))
users = response.get('data')
print(f"Total number of active users in the project: {selected_project.get('project_name')}: ", len(users))

In [ ]:
for u in users:
    # check if ceph user exists
    try:
        user_entity = f"client.{u.get('bastion_login')}"
        response = ceph_api.export_users(cluster=selected_cluster, entities=[user_entity])
        key_rings = response.get('clusters', {}).get(selected_cluster, {})
        apply_changes = False
        if user_entity not in key_rings:
            tmpl_caps = [
                {"entity": "mon", "cap": "allow r fsname={fs}"},
                {"entity": "mds", "cap": "allow rw fsname={fs} path={path}"},
                {"entity": "osd", "cap": "allow rw tag cephfs data={fs}"},
                {"entity": "osd", "cap": "allow rw tag cephfs metadata={fs}"},
            ]
            apply_changes = True
        else:
            print(f"User entity: {user_entity} already exists")
            # TODO add to existing permissions
            blob = key_rings.get(user_entity)
            text = _unescape_keyring_blob(blob)
            caps_now = extract_caps_by_entity(text)
            mds_now = caps_now.get("mds", "")
            if sub_vol_name in mds_now:
                print(f"Subvolume {sub_vol_name} already exists in capabilities -  No changes needed")
            else:
                tmpl_caps = [
                {"entity": "mon", "cap": "allow r fsname={fs}"},
                {"entity": "osd", "cap": "allow rw tag cephfs data={fs}"},
                {"entity": "osd", "cap": "allow rw tag cephfs metadata={fs}"},
            ]
                if mds_now:
                    # keep current exact MDS cap as a literal (no placeholders)
                    tmpl_caps.append({"entity": "mds", "cap": mds_now})
                # add the new MDS clause via placeholders for the new context
                tmpl_caps.append({"entity": "mds", "cap": "allow rw fsname={fs} path={path}"})
                apply_changes = True

        if apply_changes:
            response = ceph_api.apply_user_for_multiple_subvols(
                cluster=selected_cluster,
                user_entity=user_entity,
                template_capabilities=tmpl_caps,
                contexts=[
                    ("CEPH-FS-01", sub_vol_name, group_name),
                ]
            )
            print(json.dumps(response, indent=4, sort_keys=True))
    except ApiError as e:
        print("API error during export_users:", e)

In [24]:
user_entity = f"client.kthare10_0011904101"
#user_entity = "client.stealey_0000242181"
selected_cluster="east"
response = ceph_api.export_users(cluster=selected_cluster, entities=[user_entity])
print(json.dumps(response, indent=4, sort_keys=True))

/Users/kthare10/renci/code/fabric/1.6/fabric_ceph/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '23.134.232.211'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{
    "clusters": {
        "east": {}
    },
    "size": 1,
    "status": 200,
    "type": "keyrings"
}


## Minimal conf to access and mount

In [ ]:
try:
    response = ceph_api.cluster_minimal_confs()
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during cluster_minimal_confs:", e)

## Delete Sub volumes

In [ ]:
try:
    response = ceph_api.delete_subvolume(cluster=selected_cluster, vol_name=VOL_NAME, subvol_name=sub_vol_name)
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during delete_subvolume:", e)

## Delete User

In [ ]:

try:
    response = ceph_api.delete_user(cluster=selected_cluster, entity=user_entity)
    print(json.dumps(response, indent=4, sort_keys=True))
except ApiError as e:
    print("API error during delete_user:", e)